# Cell-level classifiers using frozen foundation-model features (DinoBloom-S, SubCell)

Reviewer response for **Supp. Fig. 2e/f**: *"How do other feature extractors, also
retrained single-cell foundation models like DinoBloom or SubCell, perform?"*

We use both models **frozen** as feature extractors on our single-channel 32×32
chromatin crops (no finetuning / no training from scratch — that is out of scope),
save the embeddings, and train the same leave-one-plate-out `FeatClassifier` /
`MultiFeatClassifier` used for the pretrained DINO-ViT panels. This mirrors
`cell_level_models_pretrained_dino.ipynb` so the comparison is apples-to-apples.

**Two environments (like the DINO notebook):**
- **Section 1 — feature extraction** runs in the dedicated `pbmc5-fm` env
  (SubCell needs `transformers==4.45.1`; DinoBloom uses the vendored DINOv2 under
  `foundation_models/vendor/`, which — unlike the current torch.hub DINOv2 — is
  py3.9 compatible). Features are written to `bundled_data/` and reloaded later.
- **Section 2 — training/eval** runs in the project env `new-img-39`.

Model checkpoints and the vendored model code live under
`foundation_models/` (in-repo, with provenance headers) and
`/ewsc/hschluet/pbmc5/foundation_models/` (weights).

## 1. Feature extraction  *(run with the `pbmc5-fm` env)*

Preprocessing was verified against each model's own source code:

- **DinoBloom-S** (DINOv2 ViT-S/14, CLS token → 384-d): grayscale→3ch, resize to
  224 (bicubic), ImageNet mean/std. The raw crop is used (like the existing
  DINO-ViT path) so only the extractor differs.
- **SubCell `bg`** (2-channel DNA+Protein ViT, gated-attention pool → 1536-d): mask
  the nucleus, then place it into a 640×640 frame by **integer bilinear upsampling**
  of the 32px crop — ×10 for the nucleus channel (b) and ×20 for the protein
  channel (g, an exact 32→640 upsample) — zero-padded/centered, then global
  min-max to [0,1] (matches SubCellPortable `min_max_norm_fn`). Our single chromatin
  stain is duplicated into both channels at the two scales, emulating a compact
  nucleus inside a larger cell body. Integer factors avoid resampling artifacts.

In [ ]:
import os, time
import numpy as np
import torch

# Notebooks are run from the repo root (like the other notebooks), so the
# in-repo `foundation_models` package imports directly.
import foundation_models as fm

DATA = '/ewsc/hschluet/pbmc5/bundled_data/'
CKPT_DB = '/ewsc/hschluet/pbmc5/foundation_models/dinobloom/DinoBloom-S.pth'
CKPT_SC = '/ewsc/hschluet/pbmc5/foundation_models/subcell/DNA-Protein_ViT-ProtS-Pool.pth'

device = 'cuda:0'
plates = list(range(1, 17))

In [ ]:
# Extract + save per plate. SubCell runs under bf16 autocast (cosine ~1.0 vs fp32)
# because eager attention at 640x640 (1601 tokens) is heavy. Resumable: skips any
# (plate, model) whose output already exists.
#
# For the paper this was executed via the background runner
#   /ewsc/hschluet/pbmc5/foundation_models/run_extract.py
# split across GPUs 1/2/5 (~a few hours). This cell reproduces the same logic.

def extract(plates, device, bs_db=256, bs_sc=64):
    db = fm.build_dinobloom_s(CKPT_DB, device)
    sc = fm.build_subcell_bg(CKPT_SC, device)
    for p in plates:
        imgs = torch.load(f'{DATA}plate_{p}_imgs.pt').float()
        masks = torch.load(f'{DATA}plate_{p}_masks.pt').float()
        n = len(imgs)

        db_out = f'{DATA}plate_{p}_dinobloom_feats.pt'
        if not os.path.exists(db_out):
            z = np.zeros((n, fm.DINOBLOOM_S_DIM), dtype=np.float32)
            for i in range(0, n, bs_db):
                x = fm.dinobloom_preprocess(imgs[i:i+bs_db]).to(device)
                with torch.no_grad():
                    z[i:i+len(x)] = db(x).float().cpu().numpy()
            torch.save(z, db_out)

        sc_out = f'{DATA}plate_{p}_subcell_feats.pt'
        if not os.path.exists(sc_out):
            z = np.zeros((n, fm.SUBCELL_BG_DIM), dtype=np.float32)
            for i in range(0, n, bs_sc):
                x = fm.subcell_preprocess(imgs[i:i+bs_sc], masks[i:i+bs_sc]).to(device)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    z[i:i+len(x)] = fm.subcell_embed(sc, x).float().cpu().numpy()
            torch.save(z, sc_out)
        print(f'plate {p} done (N={n})', flush=True)

# extract(plates, device)   # uncomment to run here instead of the background runner

### SubCell input composition (sanity check)

The 2-channel input SubCell actually receives: a compact **blue nucleus** (b, ×10)
inside a larger **green protein** blob (g, ×20). Uses the exact `subcell_preprocess`
from the extraction path.

In [ ]:
import matplotlib.pyplot as plt

_imgs = torch.load(f'{DATA}plate_1_imgs.pt').float()
_masks = torch.load(f'{DATA}plate_1_masks.pt').float()
_frac = _masks.reshape(len(_masks), -1).mean(1)
_ok = torch.where((_frac > 0.12) & (_frac < 0.45))[0]
_picks = [_ok[int(q * (len(_ok) - 1))] for q in torch.linspace(0.1, 0.9, 4)]

fig, axes = plt.subplots(len(_picks), 4, figsize=(8, 2 * len(_picks)))
for r, ci in enumerate(_picks):
    x = fm.subcell_preprocess(_imgs[ci:ci+1], _masks[ci:ci+1])[0]   # (2,640,640)
    nuc, prot = x[0], x[1]
    ov = torch.stack([torch.zeros_like(nuc), prot, nuc], dim=-1).clamp(0, 1)  # R,G,B
    axes[r, 0].imshow(_imgs[ci] * _masks[ci], cmap='gray')
    axes[r, 1].imshow(nuc, cmap='gray', vmin=0, vmax=1)
    axes[r, 2].imshow(prot, cmap='gray', vmin=0, vmax=1)
    axes[r, 3].imshow(ov.numpy())
    axes[r, 0].set_ylabel(f'cell {int(ci)}', fontsize=8)
    if r == 0:
        for j, t in enumerate(['masked 32px', 'nucleus ch (b, x10)', 'protein ch (g, x20)', 'overlay']):
            axes[0, j].set_title(t, fontsize=9)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
fig.suptitle('SubCell 2-channel input (bilinear integer upsample)', fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.97])

## 2. Leave-one-plate-out classifiers  *(run with the `new-img-39` env)*

Reload the saved features via `PlateDataset` and train the same classifiers used
for the DINO-ViT panels, once per model (`dinobloom` → 384-d, `subcell` → 1536-d):

- **Binary** healthy-vs-cancer (`FeatClassifier`), plates 1–16 → Supp. Fig. 2e.
- **4-way** group classification (`MultiFeatClassifier`), plates 3–12 → Supp. Fig. 2f.

Checkpoints go to the usual `outdir` with `_pretrained_{tag}` names; leave-one-out
per-cell predictions are written to `results/` for the figure notebook to plot.

In [ ]:
import importlib
import data, util, models, training
for m in (data, util, models, training):
    importlib.reload(m)

from data import PlateDataset
from util import torch_random_choice
from models import FeatClassifier, MultiFeatClassifier
from training import train_model

import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
from tqdm import tqdm

device = 'cuda:0'
OUTDIR = '/ewsc/hschluet/models/pbmc5/revision_rerun/'
os.makedirs('results', exist_ok=True)

In [ ]:
data = PlateDataset(list(range(1, 17)), load_imgs=True, load_dinobloom=True, load_subcell=True)
FEATURES = {'dinobloom': (data.dinobloom_zs, 384),
            'subcell':   (data.subcell_zs, 1536)}
print({k: v[0].shape for k, v in FEATURES.items()})

In [ ]:
# --- Binary healthy-vs-cancer: plates 1-16 (Supp. Fig. 2e) ---
use_plates_bin = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16])
cdf = data.info[data.info['plate'].isin(use_plates_bin)].groupby(['patient', 'time'])['cell'].count().reset_index()
cdf = cdf[cdf['cell'] > 100]
p01s_bin = cdf[(cdf['time'] == 1) | (cdf['time'] == 0)]['patient'].unique()


def healthy_vs_cancer_bagloader(feats, bag_size=50, use_patients=None, use_times=[0, 1],
                                use_plates=use_plates_bin, device=device):
    use_patients = p01s_bin if use_patients is None else use_patients
    use_idx = torch.argwhere(torch.from_numpy((data.info['plate'].isin(use_plates) & data.info['time'].isin(use_times)).values)).flatten()
    use_data = torch.from_numpy(feats[use_idx]).to(device)
    plate_pats = np.concatenate(data.info.groupby(['plate'])['patient'].unique().loc[use_plates].values)
    use_patients = use_patients[np.isin(use_patients, plate_pats)]
    pat_groups = data.info.groupby(['time'])['patient'].unique().map(lambda x: x[np.isin(x, use_patients)])
    pat_healthy, pat_cancer = pat_groups[0], pat_groups[1]
    pat_lut = {pat: torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == pat).values)).flatten().to(device)
               for pat in use_patients}
    pat_min = min(len(pat_healthy), len(pat_cancer))
    while True:
        xs, labels = [], []
        for lab, pats in zip([0, 1], [pat_cancer, pat_healthy]):
            for pat in np.random.choice(pats, size=pat_min, replace=False):
                xs.append(use_data[torch_random_choice(pat_lut[pat], size=bag_size)])
                labels.append(lab * torch.ones(bag_size))
        labels = torch.cat(labels).float().to(device)
        xs = torch.cat(xs).float()
        rand_idx = torch.randperm(len(xs))
        for left in range(0, bag_size * 2 * pat_min, bag_size):
            yield xs[rand_idx[left:left+bag_size]], labels[rand_idx[left:left+bag_size]]


def test_healthy_vs_cancer_bagloader(feats, bag_size=100, use_patients=None, use_times=[0, 1],
                                     use_plates=use_plates_bin, device=device, transform=T.CenterCrop(28)):
    use_patients = p01s_bin if use_patients is None else use_patients
    use_idx = torch.argwhere(torch.from_numpy((data.info['plate'].isin(use_plates) & data.info['time'].isin(use_times)).values)).flatten()
    use_data = torch.from_numpy(feats[use_idx]).to(device)
    use_imgs = transform(data.imgs[use_idx].to(device))
    pat_plates = data.info.groupby(['patient'])['plate'].unique()
    for pat in use_patients:
        if not np.isin(pat_plates[pat], use_plates).any():
            continue
        pat_idx = torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == pat).values)).flatten().to(device)
        idx = torch_random_choice(pat_idx, size=bag_size)
        label = 1 if pat[0] == 'H' else 0
        yield use_data[idx].float(), use_imgs[idx].float(), torch.tensor(label).to(device), pat, ('healthy' if pat[0] == 'H' else 'cancer')


def eval_binary(name, loader, input_dim):
    model = FeatClassifier(input_dim=input_dim).to(device)
    model.load_state_dict(torch.load(f'{OUTDIR}{name}_model.pt', weights_only=True, map_location=device))
    model.eval()
    np.random.seed(1232412); torch.manual_seed(124514); torch.cuda.manual_seed_all(13513)
    rows = {'lab': [], 'pred': [], 'i': [], 'pat': [], 'group': []}
    for i, (z, _bag, lab, pat, group) in enumerate(loader):
        with torch.no_grad():
            _, pred, _ = model(z.to(device))
        rows['i'] += [i] * len(pred); rows['pat'] += [pat] * len(pred); rows['group'] += [group] * len(pred)
        rows['lab'] += [lab.item()] * len(pred); rows['pred'] += list(pred.cpu().numpy())
    return pd.DataFrame(rows)

In [ ]:
# Train binary leave-one-plate-out for both feature sets, then eval on the held-out plate.
for tag, (feats, dim) in FEATURES.items():
    dfs = []
    for plate in use_plates_bin:
        name = f'1_16_t01_healthy_cancer_without_plate_{plate}_by_cell_pretrained_{tag}'
        train_loader = healthy_vs_cancer_bagloader(feats, use_patients=p01s_bin[~np.isin(p01s_bin, data.info.groupby('plate')['patient'].unique().loc[plate])], use_plates=use_plates_bin[use_plates_bin != plate].copy())
        model = FeatClassifier(input_dim=dim)
        train_model(model, bag_loader=train_loader, num_iter=100_000, lr=1e-4, device=device,
                    fname=name, plot=False, save_model=True, seed=12341)
        df = eval_binary(name, test_healthy_vs_cancer_bagloader(feats, use_plates=[plate]), dim)
        df['plate'] = plate
        dfs.append(df)
    res = pd.concat(dfs, ignore_index=True)
    res.to_csv(f'results/foundation_{tag}_healthy_cancer_HN_leave_one_out.csv', index=False)
    print(tag, 'binary done ->', res.shape)

In [ ]:
# --- 4-way group classification: plates 3-12 (Supp. Fig. 2f) ---
use_plates_grp = np.array([3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
use_groups = ['healthy', 'H&N cancer', 'CNS-Meningioma', 'Chordoma/Chondrosarcoma']
group_map = {'healthy': 0, 'H&N cancer': 1, 'CNS-Meningioma': 2, 'Chordoma/Chondrosarcoma': 3}

cdf = data.info[data.info['group'].isin(use_groups) & data.info['plate'].isin(use_plates_grp)].groupby(['patient', 'time'])['cell'].count().reset_index()
cdf = cdf[cdf['cell'] > 100]
p01s_grp = cdf[(cdf['time'] == 1) | (cdf['time'] == 0)]['patient'].unique()


def group_bagloader(feats, bag_size=50, use_patients=None, use_times=[0, 1], use_plates=use_plates_grp, device=device):
    use_patients = p01s_grp if use_patients is None else use_patients
    use_idx = torch.argwhere(torch.from_numpy((data.info['plate'].isin(use_plates) & data.info['time'].isin(use_times)).values)).flatten()
    use_data = torch.from_numpy(feats[use_idx]).to(device)
    plate_pats = np.concatenate(data.info.groupby(['plate'])['patient'].unique().loc[use_plates].values)
    use_patients = use_patients[np.isin(use_patients, plate_pats)]
    pat_groups = data.info.groupby(['group'])['patient'].unique().map(lambda x: x[np.isin(x, use_patients)]).loc[use_groups]
    pat_lut = {pat: torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == pat).values)).flatten().to(device)
               for pat in use_patients}
    pat_min = pat_groups.map(len).min()
    while True:
        labels, xs = [], []
        for label in use_groups:
            for pat in np.random.choice(pat_groups[label], size=pat_min, replace=False):
                xs.append(use_data[torch_random_choice(pat_lut[pat], size=bag_size)])
                labels.extend([group_map[label]] * bag_size)
        labels = torch.tensor(labels).to(device)
        xs = torch.cat(xs).float()
        rand_idx = torch.randperm(len(xs))
        for left in range(0, bag_size * len(use_groups) * pat_min, bag_size):
            yield xs[rand_idx[left:left+bag_size]], labels[rand_idx[left:left+bag_size]]


def test_group_bagloader(feats, bag_size=100, use_patients=None, use_times=[0, 1], use_plates=use_plates_grp,
                         device=device, transform=T.CenterCrop(28)):
    use_patients = p01s_grp if use_patients is None else use_patients
    use_idx = torch.argwhere(torch.from_numpy((data.info['plate'].isin(use_plates) & data.info['time'].isin(use_times)).values)).flatten()
    use_data = torch.from_numpy(feats[use_idx]).to(device)
    use_imgs = transform(data.imgs[use_idx].to(device))
    pat_groups = data.info.groupby(['patient'])['group'].max()
    pat_plates = data.info.groupby(['patient'])['plate'].unique()
    for pat in use_patients:
        if not np.isin(pat_plates[pat], use_plates).any():
            continue
        pat_idx = torch.argwhere(torch.from_numpy((data.info.loc[use_idx]['patient'] == pat).values)).flatten().to(device)
        idx = torch_random_choice(pat_idx, size=bag_size)
        label = pat_groups[pat]
        yield use_data[idx].float(), use_imgs[idx].float(), torch.tensor(group_map[label]).to(device), pat, label


def eval_multi(name, loader, input_dim, classes):
    model = MultiFeatClassifier(classes=classes, input_dim=input_dim).to(device)
    model.load_state_dict(torch.load(f'{OUTDIR}{name}_model.pt', weights_only=True, map_location=device))
    model.eval()
    np.random.seed(1232412); torch.manual_seed(124514); torch.cuda.manual_seed_all(13513)
    rows = {'lab': [], 'pred': [], 'i': [], 'pat': [], 'group': []}
    for i, (z, _bag, lab, pat, group) in enumerate(loader):
        with torch.no_grad():
            _, pred, _ = model(z.to(device))
        rows['i'] += [i] * len(pred); rows['pat'] += [pat] * len(pred); rows['group'] += [group] * len(pred)
        rows['lab'] += [lab.item()] * len(pred); rows['pred'] += list(pred.cpu().numpy())
    return pd.DataFrame(rows)

In [ ]:
# Train 4-way leave-one-plate-out for both feature sets, then eval on the held-out plate.
for tag, (feats, dim) in FEATURES.items():
    dfs = []
    for plate in use_plates_grp:
        name = f'3_12_t01_healthy_cancer_without_plate_{plate}_by_cell_pretrained_{tag}'
        train_loader = group_bagloader(feats, use_patients=p01s_grp[~np.isin(p01s_grp, data.info.groupby('plate')['patient'].unique().loc[plate])], use_plates=use_plates_grp[use_plates_grp != plate].copy())
        model = MultiFeatClassifier(classes=len(use_groups), input_dim=dim)
        train_model(model, bag_loader=train_loader, num_iter=400_000, lr=1e-4, device=device,
                    fname=name, plot=False, save_model=True, seed=12341)
        df = eval_multi(name, test_group_bagloader(feats, use_plates=[plate]), dim, len(use_groups))
        df['plate'] = plate
        dfs.append(df)
    res = pd.concat(dfs, ignore_index=True)
    res.to_csv(f'results/foundation_{tag}_groups_leave_one_out.csv', index=False)
    print(tag, 'multiclass done ->', res.shape)